# 学习率

上一章，我们用梯度下降法进行了第一次模型训练。方向是对的，但一步把参数从 `[0.5, 0.5]` 更新到了 `[6854, 14147]`，损失值随之爆炸，模型训练失败。

问题的根源在于：**梯度只告诉我们下坡的方向，却不能决定步子应该迈多大。**

梯度很大，只说明当前这个方向的坡度很陡，并不意味着应该迈一大步。步子太大，会直接越过山谷，冲到对面的坡上，反而比出发点更高。这种现象称为**发散**（Divergence）。

---

解决办法是引入一个比例系数，把每次更新的步幅等比例缩小，这个系数称为**学习率**（Learning Rate），记作 $\eta$。

加入学习率后，梯度下降的更新公式变为：

$$
w_{\text{new}} = w_{\text{old}} - \eta \cdot \frac{\partial L}{\partial w}
$$
$$
b_{\text{new}} = b_{\text{old}} - \eta \cdot \frac{\partial L}{\partial b}
$$

学习率需要仔细选择：

* **太大**：更新步幅过大，损失值左右震荡，甚至发散；
* **太小**：更新步幅过小，损失值下降极慢，需要大量训练才能收敛；
* **合适**：损失值稳定地逐步下降，最终收敛到最优解附近。

## 超参数

学习率是我们遇到的第一个**超参数**（Hyperparameter）。

模型参数（权重和偏置）是模型**自动学习**的对象。每次训练后，它们会根据梯度自动更新。

超参数则不同，它们是用来控制训练过程的。超参数**不会被模型训练自动调整**，而是需要我们在训练开始前根据经验和实验手动设定。学习率控制的是训练步幅，后续章节还会遇到其他超参数。

不断测试、调整超参数以获得最佳训练效果的过程，称为**调参**（Hyperparameter Tuning）。

In [1]:
import numpy as np

## 数据

继续使用小明的冰激凌数据：温度 28.1°C、湿度 58%，实际销量 165 个。

### 特征、标签

In [2]:
feature = np.array([28.1, 58.0])
label = np.array([165])

## 模型

模型框架与前两章相同。本章只需对反向函数做一处改动：加入学习率。

### 参数：权重、偏置

In [3]:
weight = np.ones((1, 2)) / 2
bias = np.zeros(1)

### 推理函数

In [4]:
def forward(x, w, b):
    return x @ w.T + b

### 损失函数（均方误差）

In [5]:
def mse_loss(p, y):
    return np.mean(np.square(y - p))

### 梯度函数

In [6]:
def gradient(p, y):
    return -2 * (y - p)

### 反向函数

在上一章的反向函数基础上，加入学习率参数 `lr`，用它等比例缩小每次更新的步幅：

$$
w \leftarrow w - \eta \cdot \delta \cdot x
$$
$$
b \leftarrow b - \eta \cdot \delta
$$

In [7]:
def backward(x, d, w, b, lr):
    w -= d * x * lr
    b -= d * lr
    return w, b

## 训练

### 超参数：学习率

学习率没有放之四海而皆准的最优值，需要根据模型和数据反复实验。实践中常用的策略是：

1. 从一个标准起点开始——通常是 `0.001`；
2. 运行一次训练，观察损失值的变化：
   * 若损失值大幅震荡或发散 → 学习率太大，缩小 10 倍；
   * 若损失值几乎不变 → 学习率太小，放大 10 倍；
   * 若损失值稳定下降 → 学习率合适，继续观察。

在本例中，特征值（温度约 28、湿度约 58）和标签值（约 165）的数值范围都较大，导致梯度本身就很大（上一章算出 $\delta$ 大约是 -244）。直接套用 `0.001` 仍会发散，因此我们选择更保守的 `0.00001`。

In [8]:
LEARNING_RATE = 0.00001

### 前向传播

In [9]:
prediction = forward(feature, weight, bias)
print(f'prediction: {prediction}')

prediction: [43.05]


### 计算损失

In [10]:
loss = mse_loss(prediction, label)
print(f'loss: {loss:.4f}')

loss: 14871.8025


### 梯度计算

In [11]:
delta = gradient(prediction, label)
print(f'delta: {delta}')

delta: [-243.9]


### 反向传播

这次，梯度乘以学习率 `0.00001` 后再更新参数，步幅被缩小到原来的万分之一。

In [12]:
weight, bias = backward(feature, delta, weight, bias, LEARNING_RATE)
print(f'weight: {weight}')
print(f'bias:   {bias}')

weight: [[0.5685359 0.641462 ]]
bias:   [0.002439]


权重从 `[0.5, 0.5]` 更新到 `[0.569, 0.641]`，偏置从 `0` 更新到 `0.0024`。更新方向与上一章相同（参数增大），但幅度从数千降到了零点几，终于在合理范围内了。

## 验证

### 推理

In [13]:
prediction = forward(feature, weight, bias)
print(f'prediction: {prediction}')

prediction: [53.18309379]


### 评估

In [14]:
loss_after = mse_loss(prediction, label)
print(f'loss: {loss_after:.4f}')

loss: 12503.0205


引入学习率后，一次训练使损失值从 `14,871` 降至 `12,503`，下降了约 16%。

预测值也从 `43` 提升到了 `53`，朝真实值 165 的方向移动。训练方向正确，但还差得远。

这很正常：学习率的使用，控制一次训练只能迈出一小步。因此需要多走几步，才能到达目的地。这正是下一章要解决的问题。

## 课后练习

尝试以下不同的学习率，每次只运行一次训练，观察结果：

| 学习率 | 预期现象 |
|--------|---------|
| `0.1` | 发散（损失值爆炸）|
| `0.001` | 发散或严重震荡 |
| `0.0001` | 损失值大幅下降 |
| `0.00001` | 损失值适度下降（本章结果）|
| `0.000001` | 损失值几乎不变 |

实际运行验证一下，结果是否和预期一致？